<a href="https://colab.research.google.com/github/venkata18167/CSA6102-DIGITAL-FORENSICS-AND-CYBERCRIME-INVESTIGATION/blob/main/14_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
 import re
from urllib.parse import urlparse
import pandas as pd

print("="*70)
print("          PHISHING EMAIL ANALYSIS")
print("="*70)

email = {
    "From": "PayPal Support <support@paypa1.com>",
    "Reply-To": "help@secure-paypal-login.com",
    "Return-Path": "service@paypa1.com",
    "Subject": "URGENT: Verify your PayPal Account",
    "Body": """
Dear Customer,

Your PayPal account has been temporarily suspended.

To restore your account, verify your identity immediately.

Click the secure link below:

http://paypal-login-security.xyz/login

OR

https://secure-paypal-login.com/verify

Failure to verify within 24 hours will permanently suspend your account.

Regards,
PayPal Security Team
"""
}

print("\nEMAIL HEADER")
print("-"*70)

print("From       :", email["From"])
print("Reply-To   :", email["Reply-To"])
print("ReturnPath :", email["Return-Path"])
print("Subject    :", email["Subject"])

print("\nEMAIL BODY")
print("-"*70)
print(email["Body"])

match = re.search(r'<(.*?)>', email["From"])

if match:
    sender_email = match.group(1)
else:
    sender_email = email["From"]

sender_domain = sender_email.split("@")[-1]

print("\nSENDER DETAILS")
print("-"*70)

print("Sender Email :", sender_email)
print("Sender Domain:", sender_domain)

urls = re.findall(r'https?://[^\s]+', email["Body"])

print("\nEXTRACTED URLS")
print("-"*70)

for u in urls:
    print(u)

print("\nURL ANALYSIS")
print("-"*70)

keywords = [
    "login",
    "verify",
    "secure",
    "account",
    "bank",
    "update",
    "signin"
]

table = []

risk_score = 0

for url in urls:

    parsed = urlparse(url)

    domain = parsed.netloc

    suspicious = False

    reasons = []

    if ".xyz" in domain:
        suspicious = True
        reasons.append("Unknown .xyz domain")

    if "-" in domain:
        suspicious = True
        reasons.append("Hyphenated domain")

    if any(word in url.lower() for word in keywords):
        suspicious = True
        reasons.append("Contains phishing keywords")

    if len(domain) > 20:
        suspicious = True
        reasons.append("Long domain")

    if suspicious:
        risk_score += 20

    table.append({
        "URL": url,
        "Domain": domain,
        "Status": "Suspicious" if suspicious else "Legitimate",
        "Reason": ", ".join(reasons)
    })

df = pd.DataFrame(table)

print(df)

print("\nSENDER VERIFICATION")
print("-"*70)

trusted_domain = "paypal.com"

spoof = False

reason = []

if sender_domain != trusted_domain:
    spoof = True
    reason.append("Sender domain differs from trusted domain")

if "1" in sender_domain:
    spoof = True
    reason.append("Possible character replacement (1 instead of l)")

if spoof:
    risk_score += 30

print("Trusted Domain :", trusted_domain)
print("Actual Domain  :", sender_domain)
print("Spoof Detected :", spoof)

if spoof:
    print("Reason(s):")

    for r in reason:
        print("-", r)
print("\nSUBJECT ANALYSIS")
print("-"*70)

subject = email["Subject"].lower()

subject_words = ["urgent","verify","account","suspended"]

for word in subject_words:

    if word in subject:
        print(word,"--> Suspicious Keyword")
        risk_score += 5

print("\nBODY ANALYSIS")
print("-"*70)

body = email["Body"].lower()

body_words = [
    "verify",
    "urgent",
    "suspend",
    "click",
    "secure"
]

for word in body_words:

    if word in body:
        print(word,"--> Found")
        risk_score += 2


if risk_score > 100:
    risk_score = 100

print("\nPHISHING RISK SCORE")
print("-"*70)

print("Risk Score :", risk_score,"/100")

if risk_score >= 80:
    print("Risk Level : HIGH")

elif risk_score >= 50:
    print("Risk Level : MEDIUM")

else:
    print("Risk Level : LOW")

print("\nINDICATORS OF COMPROMISE (IoCs)")
print("-"*70)

print("Sender Email :", sender_email)

print("Sender Domain:", sender_domain)

print("\nSuspicious URLs")

for row in table:

    if row["Status"]=="Suspicious":
        print("-",row["URL"])


print("\nSUMMARY STATISTICS")
print("-"*70)

print("Total URLs          :",len(urls))

print("Suspicious URLs     :",sum(x["Status"]=="Suspicious" for x in table))

print("Spoofed Sender      :",spoof)

print("Risk Score          :",risk_score)

print("\nExperiment Completed Successfully.")

          PHISHING EMAIL ANALYSIS

EMAIL HEADER
----------------------------------------------------------------------
From       : PayPal Support <support@paypa1.com>
Reply-To   : help@secure-paypal-login.com
ReturnPath : service@paypa1.com
Subject    : URGENT: Verify your PayPal Account

EMAIL BODY
----------------------------------------------------------------------

Dear Customer,

Your PayPal account has been temporarily suspended.

To restore your account, verify your identity immediately.

Click the secure link below:

http://paypal-login-security.xyz/login

OR

https://secure-paypal-login.com/verify

Failure to verify within 24 hours will permanently suspend your account.

Regards,
PayPal Security Team


SENDER DETAILS
----------------------------------------------------------------------
Sender Email : support@paypa1.com
Sender Domain: paypa1.com

EXTRACTED URLS
----------------------------------------------------------------------
http://paypal-login-security.xyz/login
https